In [1]:
print("Hello, world!...")

Hello, world!...


In [ ]:
uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-commuinity

In [1]:
! uv pip install langchain langchain-huggingface langchain-community rapidocr-onnxruntime python-dotenv sentence-transformers

/bin/bash: line 1: uv: command not found


In [15]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

### Data Ingestion

from langchain.document_loaders import TextLoader

In [6]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("/home/ahmadtigress/Documents/rag_project/data/Agentic AI.txt", encoding="utf8")
documents = loader.load()

print(f"Successfully loaded {len(documents)} document(s).")

/home/ahmadtigress/Documents/rag_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Successfully loaded 1 document(s).


In [7]:
documents[0].page_content[:500]      # print the first 500 characters of the first document

'Understanding Agentic AI\n\nAgentic AI: Understanding the Next Frontier of Intelligent SystemsSlide 1: Title SlideTitle: Agentic AI: Understanding Agentic AISubtitle: Moving From Passive Automation to Autonomous SubsystemsPresenter: [Your Name/Title]Context: Technology Presentation & Executive BriefingSlide 2: The Evolution of Artificial IntelligenceShifting ParadigmsArtificial Intelligence is undergoing a fundamental shift in how it interacts with human intent, transitioning from static generatio'

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)

In [10]:
text_chunks = text_splitter.split_documents(documents)

In [11]:
text_chunks

[Document(metadata={'source': '/home/ahmadtigress/Documents/rag_project/data/Agentic AI.txt'}, page_content='Understanding Agentic AI'),
 Document(metadata={'source': '/home/ahmadtigress/Documents/rag_project/data/Agentic AI.txt'}, page_content='Agentic AI: Understanding the Next Frontier of Intelligent SystemsSlide 1: Title SlideTitle: Agentic AI: Understanding Agentic AISubtitle: Moving From Passive Automation to Autonomous'),
 Document(metadata={'source': '/home/ahmadtigress/Documents/rag_project/data/Agentic AI.txt'}, page_content='to Autonomous SubsystemsPresenter: [Your Name/Title]Context: Technology Presentation & Executive BriefingSlide 2: The Evolution of Artificial IntelligenceShifting ParadigmsArtificial Intelligence is'),
 Document(metadata={'source': '/home/ahmadtigress/Documents/rag_project/data/Agentic AI.txt'}, page_content='Intelligence is undergoing a fundamental shift in how it interacts with human intent, transitioning from static generation to active execution.Gene

In [12]:
! uv pip install pip faiss-cpu
print("faiss installed...")

Using Python 3.12.3 environment at: /home/ahmadtigress/Documents/rag_project/.venv
Resolved 4 packages in 11.71s                                        
⠙ Preparing packages... (0/2)                                                   
⠹ Preparing packages... (0/2)-------------------     0 B/1.73 MiB            
⠹ Preparing packages... (0/2)-------------------     0 B/1.73 MiB            
⠹ Preparing packages... (0/2)------------------- 16.00 KiB/1.73 MiB          
⠹ Preparing packages... (0/2)------------------- 32.00 KiB/1.73 MiB          
⠹ Preparing packages... (0/2)------------------- 41.33 KiB/1.73 MiB          
⠹ Preparing packages... (0/2)------------------- 41.33 KiB/1.73 MiB          
pip                  ------------------------------ 41.33 KiB/1.73 MiB
⠸ Preparing packages... (0/2)-------------------     0 B/22.74 MiB           
pip                  ------------------------------ 41.33 KiB/1.73 MiB
faiss-cpu            ------------------------------     0 B/22.74 MiB         

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [2]:
# 2. Force the Hugging Face Hub library to show progress bars
#from huggingface_hub import configure_http_backend
import huggingface_hub

## 2. Force Hugging Face to enable live download progress bars
huggingface_hub.enable_progress_bars()

print("Starting download process for 'all-MiniLM-L6-v2'...")
print("Look below for the live progress bar 👇")

AttributeError: No huggingface_hub attribute enable_progress_bars

In [ ]:
# 1. LOAD THE SEARCH ENGINE (Lightweight - ~500MB RAM)
# This model converts words to math vectors. 
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={'device': 'cpu'})


In [ ]:
vectore_store = FAISS.to_documents(text_chunks, embeddings)

In [ ]:
vectore_store

In [ ]:
retriever = vectore_store.as_retriever()

In [ ]:
# Perform a similarity search

query = "What is the key characteristic of Agentic AI?"
docs = vectore_store.similarity_search(query, k=4)

# Display the result
for i, doc in enumerate(docs):
    print(f"Documents {i+1}:")
    print(doc.page_content)
    print("_" * 50)

In [ ]:
from langchain.prompts import ChatPromptTemplate

template = """ You are an assistant for question-answering tasks. Use the following piece of retrieved context
to answer the question. If you do not know the answer, just say that you do not know. Use ten sentences maximum
and keep the answer concise. 

Question: {question}
Context: {context}
Answer:
"""

In [ ]:
prompt = ChatPromptTemplate.from_template(template)

In [ ]:
prompt

In [ ]:
from langchain.schema.outputparser import StrOutputParser

In [ ]:
output_parser=StrOutputParser()

In [ ]:
from transformers import pipeline
from langchain.llms import HuggingFacePipeline

# Load FLAN-T5 model
pipe = pipeline("text2text-generation", model="google/flan-t5-base")
llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Build RAG chain
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

In [ ]:
rag_chain.invoke("Tell me about agentic AI")